# qShape syllabifier — a LatinCy pipeline demo

**What this shows:** adding the experimental `syllabifier` component to a stock
LatinCy pipeline (`la_core_web_lg`) so every token carries **syllables** and a
**qShape** — a prosody-aware, per-syllable weight string over `{H, L, x}`:

| symbol | meaning |
|---|---|
| `H` | **heavy** — long vowel, diphthong, or closed by position |
| `L` | **light** — open syllable, short vowel (known) |
| `x` | **honest unknown** — open syllable whose vowel length spelling can't decide, or muta-cum-liquida |

The payoff isn't scansion for its own sake: **vowel quantity carries morphology the
letters hide** (`puella` nom. vs `puellā` abl.; `venit` pres. vs `vēnit` perf.). qShape
marks exactly the sites it *can't* resolve with `x` — and never fabricates the very
distinction a tagger exists to make. We then show two independent signals that *fill in*
those `x` sites: the **macronizer** and a new **IPA-derived gold-natura table** mined from
Wiktionary/kaikki pronunciations.


In [1]:
import json
from pathlib import Path

import spacy
import latincy_ext  # registers the "syllabifier" + "macron_morph" factories
from latincy_ext.syllabify import syllabify, qshape

nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("syllabifier")            # append after the stock pipeline
nlp.pipe_names

['enclitic_splitter',
 'tok2vec',
 'senter',
 'token_fix',
 'normer',
 'tagger',
 'morphologizer',
 'trainable_lemmatizer',
 'lookup_lemmatizer',
 'uv_normalizer',
 'harmonizer',
 'remorpher',
 'parser',
 'ner',
 'syllabifier']

## 1 · qShape on a famous line

*Arma virumque cano, Troiae qui primus ab oris* — Aeneid 1.1, as-is (no macrons).

One wrinkle first: LatinCy's `lg` model normalizes `v→u` in token **text**
(`virumque`→`uirumque`), and the syllabifier only treats consonantal *i*, not consonantal
*u/v* — so scanning `token.text` over-splits `uirum`→`u·i·rum`. But the pipeline's
`uv_normalizer` component preserves the classical spelling in **`token._.uv_normalized`**
(and the lemma in `._.uv_normalized_lemma`), so we scan off *that*.

In [2]:
def scan_form(t):
    return t._.uv_normalized or t.text        # classical v-spelling the pipeline restores

def show(doc):
    rows = [("FORM", "SYLLABLES", "qSHAPE")]
    for t in doc:
        if t.is_punct:
            continue
        f = scan_form(t)
        rows.append((f, "\u00b7".join(syllabify(f)), qshape(f)))
    w0 = max(len(r[0]) for r in rows); w1 = max(len(r[1]) for r in rows)
    for a, b, c in rows:
        print(f"{a:<{w0}}   {b:<{w1}}   {c}")

show(nlp("Arma virumque cano, Troiae qui primus ab oris"))

FORM     SYLLABLES   qSHAPE
Arma     Ar·ma       H.x
virum    vi·rum      x.H
que      que         x
cano     ca·no       x.x
Troiae   Tro·iae     H.H
qui      qui         x
primus   pri·mus     x.H
ab       ab          H
oris     o·ris       x.H


`virum` → `vi·rum` `x.H` (correct — `vĭrŭm`). Now read the `x`s as *signposts*.
`cano` → `x.x`: from the bare spelling alone the component refuses to commit either vowel —
and indeed `canō` ("I sing", 1sg pres.) and `cānō` (of *cānus*, "grey") are a real minimal
pair. `Arma` → `H.x` (the `Ar` is heavy **by position**; the final `a` stays honest);
`Troiae` → `H.H` (diphthong + consonantal-*i*).

> **Design note (fragile — to revisit):** this relies on `._.uv_normalized`, which only
> exists if `uv_normalizer` has already run. Nothing *enforces* that ordering — a user who
> adds the syllabifier earlier (or reads `token.text`) gets silently-wrong scansion with no
> error. The component should declare that dependency (e.g. `requires=[...]` / an enforced
> position) rather than trust the author to know it must come *after* the normalizer. It
> works here, but it's a seam we want to redesign, not lean on.

## 2 · Quantity carries morphology

Identical (or near-identical) spelling, different quantity → different **case** or **tense**.
qShape puts an `x` exactly on the syncretic vowel; supply the macron and it resolves.

In [3]:
def qs(word):
    return nlp(word)[0]._.qshape

pairs = [("rosa", "rosā", "nom. vs abl."),
         ("puella", "puellā", "nom. vs abl."),
         ("cano", "canō", "ambiguous vs 1sg pres.")]
print(f"{'bare':10} {'qShape':10}   {'macronized':10} {'qShape':10}   note")
print("-" * 62)
for bare, macron, note in pairs:
    print(f"{bare:10} {qs(bare):10}   {macron:10} {qs(macron):10}   {note}")

bare       qShape       macronized qShape       note
--------------------------------------------------------------
rosa       x.x          rosā       L.H          nom. vs abl.
puella     x.H.x        puellā     L.H.H        nom. vs abl.
cano       x.x          canō       L.H          ambiguous vs 1sg pres.


The macron flips the honest `x` to a definite `L`/`H` — the component treats *absence*
of a macron in a macronized context as the positive signal "short". That's the
**macronizer's** job in a full pipeline. But what if we have no macronizer, only bare text?

## 3 · Resolving `x` from IPA gold natura *(new)*

`latincy-words/scripts/extract_ipa_prosody.py` mines **Classical-Latin IPA** from the
kaikki/Wiktionary dump — `[aˈmiː.kʊs]` — where `ː` marks a long vowel and a diphthong
glide marks length too. It produces a table keyed by the **bare** form → the vowel
**natura** (`L`ong / `S`hort) per syllable, independent of spelling. Genuinely ambiguous
forms keep *all* their readings, so we can intersect them position-wise: agreeing nuclei
resolve, disagreeing nuclei stay `x`.

In [4]:
IPA_PATH = Path("../../latincy-words/data/processed/latin-forms-ipa-prosody.json")
ipa = json.load(open(IPA_PATH, encoding="utf-8"))
print(f"{len(ipa):,} bare forms with IPA-derived natura\n")

for w in ["amicus", "cano", "rosa"]:
    readings = " | ".join(f"{r['natura']} ({r['macronized']})" for r in ipa[w])
    print(f"{w:8} -> {readings}")

41,725 bare forms with IPA-derived natura

amicus   -> S.L.S (amīcus)
cano     -> S.L (canō) | L.L (cānō)
rosa     -> S.S (rosa) | S.L (rosā) | L.S (rōsa) | L.L (rōsā)


`rosa` has four readings (rosa / rosā / rōsa / rōsā) that disagree on *both* nuclei —
so IPA, like qShape, correctly leaves it `x.x`. `amicus` has a single reading `S.L.S`, and
`cano` two that agree only on the second nucleus. Now overlay natura onto qShape: an `x`
in an open syllable becomes `H` if the nucleus is long, `L` if short, and stays `x` if the
readings disagree or don't align.

In [5]:
STRIP = str.maketrans("āēīōūȳ", "aeiouy")

def resolve_with_ipa(token):
    q = token._.qshape.split(".")
    bare = token.text.translate(STRIP).lower()
    reads = [r["natura"].split(".") for r in ipa.get(bare, [])
             if len(r["natura"].split(".")) == len(q)]
    out = []
    for i, sym in enumerate(q):
        if sym == "x" and reads:
            vals = {r[i] for r in reads}
            out.append("H" if vals == {"L"} else "L" if vals == {"S"} else "x")
        else:
            out.append(sym)
    return ".".join(out)

print(f"{'form':8} {'qShape (bare)':16} {'+ IPA natura':16} outcome")
print("-" * 58)
notes = {"amicus": "fully resolved -> matches amīcus",
         "cano":   "2nd nucleus recovered; 1st honestly stays x",
         "rosa":   "readings disagree -> stays honest x"}
for w in ["amicus", "cano", "rosa"]:
    t = nlp(w)[0]
    print(f"{w:8} {t._.qshape:16} {resolve_with_ipa(t):16} {notes[w]}")

form     qShape (bare)    + IPA natura     outcome
----------------------------------------------------------
amicus   x.x.H            L.H.H            fully resolved -> matches amīcus
cano     x.x              x.H              2nd nucleus recovered; 1st honestly stays x
rosa     x.x              x.x              readings disagree -> stays honest x


## 4 · Hearing it — IPA → speech (`speaker`)

The `speaker` component turns those IPA annotations into audio. It attaches
`token._.ipa` from the same table, maps each IPA string to phonemes, and drives a
TTS backend — **carrying vowel length through to the sound**, the distinction most
Latin TTS drops. Default backend is `espeak-ng` (tiny, offline, phoneme-native);
`backend="coqui"` swaps in a neural donor voice when you want nicer timbre.

We demo on **prose**, deliberately. `speaker` is a *token-level* reader: it speaks each
word's lexical pronunciation. It does **not** scan — no elision, no cross-word positional
length, no metrical timing. Those are span-level (`mShape`) effects we haven't built, so
feeding it a verse line would imply a metrical reading it isn't making. Caesar's opening is
the honest test.

In [ ]:
import shutil
from IPython.display import Audio, display

if "speaker" not in nlp.pipe_names:
    nlp.add_pipe("speaker", config={"lookup_path": str(IPA_PATH)})

# # De Bello Gallico, Book 1, section 1 — prose, in full
DBG_1_1 = (
    "Gallia est omnis divisa in partes tres, quarum unam incolunt Belgae, "
    "aliam Aquitani, tertiam qui ipsorum lingua Celtae, nostra Galli appellantur. "
)

doc = nlp(DBG_1_1)
spk = nlp.get_pipe("speaker")

n_ipa = sum(1 for t in doc if t._.ipa)
n_tok = sum(1 for t in doc if not t.is_punct)
print(f"{n_tok} tokens, {n_ipa} with IPA from the table; rest fall back to espeak G2P.")
print("sample token IPA (classical [w] for v, ː length):")
for t in doc:
    if t.text.lower() in ("divisa", "lingua", "gallia", "belgae"):
        print(f"  {t.text:9} {t._.ipa}")

print(f"\nspeaking at {spk.rate} wpm (espeak's native ~175 is too fast for Latin)...")

if shutil.which("espeak-ng"):
    display(Audio(spk.speak(doc, out_path="caesar_ipa.wav")))
else:
    print("\n[install espeak-ng — e.g. `brew install espeak-ng` — then re-run to hear it]")

9 tokens, 6 with IPA from the table; rest fall back to espeak G2P.
sample token IPA (classical [w] for v, ː length):

speaking at 120 wpm (espeak's native ~175 is too fast for Latin)...


## Takeaways

- The `syllabifier` drops into any LatinCy pipeline in one line and annotates every token
  with `._.syllables` and `._.qshape` — **non-destructively** (it never overwrites
  `token.morph`/`pos_`).
- qShape is **honest**: `x` marks precisely the morphologically-loaded sites spelling can't
  resolve, so it's safe as a downstream model feature.
- Two independent signals close the gap: the **macronizer** (in-context) and the new
  **IPA gold-natura table** (lexical, from kaikki pronunciations). `amicus` → `x.x.H`
  becomes `L.H.H` from IPA alone, with no macronizer in the pipeline.
- The same IPA annotations drive the **`speaker`** component: `Doc` → phonemes → audio,
  with vowel length preserved — an IPA layer doubles as a TTS front-end.

**Roadmap:** wire the IPA table into `syllabifier` via `lookup_path` (position-wise
intersection resolve of `x`); an IPA-gold **eval** of qShape accuracy + a toy training
**ablation** (does qShape add signal beyond floret n-grams for Case tagging?); a span-level
`mshaper` for elision / brevis-in-longo / clausulae; and consonantal-`u/v` syllabification.
